# IPC2BNS-Verify — STEP 1: Project Setup

**Run this notebook ONCE in your first Colab session.**  
It will:
1. Mount Google Drive
2. Create the full project directory structure under `/content/drive/MyDrive/NLP_rspaper/`
3. Copy all planning docs into `docs/`
4. Write `check_progress.py` into the project root
5. Create `checkpoints/progress_state.json` with initial state
6. Verify the setup by running `check_progress.py`

---
## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted successfully.')

---
## 2. Define Project Root & Create Directory Structure

This matches the Data Management Plan exactly.

In [ ]:
import os

# ── Project root on Google Drive ──────────────────────────────────
PROJECT_ROOT = '/content/drive/MyDrive/NLP_rspaper'

# ── Full directory tree (from Data Management Plan) ───────────────
DIRECTORIES = [
    # code/
    'code/src/mapping',
    'code/src/ingestion',
    'code/src/retrieval',
    'code/src/generation',
    'code/src/verifier',
    'code/src/refresh',
    'code/src/eval',
    'code/tests',
    'code/configs',
    # data/
    'data/00_raw/india_code',
    'data/00_raw/concordance_source_pdfs',
    'data/01_cleaned',
    'data/02_ground_truth',
    'data/03_benchmark',
    'data/04_refresh_sim',
    'data/05_embeddings_index',
    # results/
    'results/stage1',
    'results/stage2',
    'results/stage3',
    'results/stage4',
    # docs, report, checkpoints
    'docs',
    'report',
    'checkpoints',
]

print(f'📁 Creating project structure under: {PROJECT_ROOT}')
print()

for d in DIRECTORIES:
    full_path = os.path.join(PROJECT_ROOT, d)
    os.makedirs(full_path, exist_ok=True)
    print(f'  ✅ {d}/')

print(f'\n🎉 All {len(DIRECTORIES)} directories created successfully.')

---
## 3. Copy Planning Docs to `docs/`

Searches for the docs in the Drive root or `required_doc_files/` subfolder.  
If not found, use the upload widget in the next cell.

In [ ]:
import shutil

PLANNING_DOCS = [
    'IPC2BNS-Verify_Research_Proposal.md',
    'IPC2BNS-Verify_Technical_Pipeline.md',
    'IPC2BNS-Verify_Data_Management_Plan.md',
    'IPC2BNS-Verify_Ground_Truth_Concordance_Runbook.md',
    'IPC2BNS-Verify_Task_Board_WBS.md',
    'IPC2BNS-Verify_Project_Management_Guide.md',
]

# Places to look for the docs
SEARCH_DIRS = [
    PROJECT_ROOT,
    f'{PROJECT_ROOT}/required_doc_files',
    '/content',
]

docs_dir = os.path.join(PROJECT_ROOT, 'docs')
found_count = 0

for doc_name in PLANNING_DOCS:
    dest = os.path.join(docs_dir, doc_name)
    if os.path.exists(dest):
        print(f'  ⏭️  {doc_name} — already in docs/')
        found_count += 1
        continue

    source = None
    for search_dir in SEARCH_DIRS:
        candidate = os.path.join(search_dir, doc_name)
        if os.path.exists(candidate):
            source = candidate
            break

    if source:
        shutil.copy2(source, dest)
        print(f'  ✅ {doc_name} — copied from {os.path.dirname(source)}')
        found_count += 1
    else:
        print(f'  ❌ {doc_name} — NOT FOUND. Upload manually.')

print(f'\n📄 {found_count}/{len(PLANNING_DOCS)} planning docs in place.')

if found_count < len(PLANNING_DOCS):
    print('\n⚠️  Missing docs — upload them via the next cell or to Drive directly.')

### 3b. Manual upload (only if docs are missing above)

In [ ]:
# Uncomment and run this cell ONLY if docs were not found above.

# from google.colab import files
# uploaded = files.upload()  # select all 6 .md files
#
# docs_dir = os.path.join(PROJECT_ROOT, 'docs')
# for filename, content in uploaded.items():
#     dest = os.path.join(docs_dir, filename)
#     with open(dest, 'wb') as f:
#         f.write(content)
#     print(f'  Uploaded → {dest}')
#
# print('Done. Re-run cell 3 to verify.')

---
## 4. Write `check_progress.py` to Project Root

In [ ]:
# Read check_progress.py content and write it to the project root

check_progress_code = '#!/usr/bin/env python3\n"""\ncheck_progress.py\n\nScans the IPC2BNS-Verify project folder and reports how much of the\nWork Breakdown Structure is actually done, based on which files/folders\nexist on disk. Run this any time to get an up-to-date status table\ninstead of manually updating a checklist.\n\nUsage:\n    python check_progress.py --root /path/to/IPC2BNS-Verify\n    python check_progress.py                      # defaults to current directory\n    python check_progress.py --write-report        # also saves results/progress_report.md\n"""\n\nimport argparse\nimport os\nfrom datetime import datetime\n\n# ---------------------------------------------------------------------------\n# Task definitions: each task is "done" if ALL of its check_paths exist\n# (and, where noted, are non-empty). Paths are relative to project root.\n# Edit this list as your actual folder/file names solidify.\n# ---------------------------------------------------------------------------\n\nTASKS = [\n    # --- Phase 0: Setup ---\n    {"phase": "0. Setup", "task": "Repo scaffolding + config system",\n     "check_paths": ["code/src", "code/configs"]},\n    {"phase": "0. Setup", "task": "India Code raw text downloaded",\n     "check_paths": ["data/00_raw/india_code"]},\n    {"phase": "0. Setup", "task": "Concordance source PDF(s) collected",\n     "check_paths": ["data/00_raw/concordance_source_pdfs"]},\n    {"phase": "0. Setup", "task": "Data Management Plan written",\n     "check_paths": ["docs/IPC2BNS-Verify_Data_Management_Plan.md"]},\n\n    # --- Phase 1: Mapping Module ---\n    {"phase": "1. Mapping Module", "task": "Ground-truth concordance table finalized",\n     "check_paths": ["data/02_ground_truth/concordance_v1.csv"]},\n    {"phase": "1. Mapping Module", "task": "Concordance validation report reviewed",\n     "check_paths": ["data/02_ground_truth/validation_report.csv"]},\n    {"phase": "1. Mapping Module", "task": "Deterministic lookup function implemented",\n     "check_paths": ["code/src/mapping/lookup.py"]},\n    {"phase": "1. Mapping Module", "task": "Query normalizer implemented",\n     "check_paths": ["code/src/mapping/normalizer.py"]},\n    {"phase": "1. Mapping Module", "task": "Mapping module unit tests",\n     "check_paths": ["code/tests/test_concordance.py"]},\n\n    # --- Phase 2: Ingestion & Retrieval ---\n    {"phase": "2. Ingestion & Retrieval", "task": "Section-level chunker implemented",\n     "check_paths": ["code/src/ingestion/chunker.py"]},\n    {"phase": "2. Ingestion & Retrieval", "task": "Cleaned section corpus produced",\n     "check_paths": ["data/01_cleaned/ipc_sections.jsonl", "data/01_cleaned/bns_sections.jsonl"]},\n    {"phase": "2. Ingestion & Retrieval", "task": "Benchmark question set drafted (dev)",\n     "check_paths": ["data/03_benchmark/benchmark_dev.csv"]},\n    {"phase": "2. Ingestion & Retrieval", "task": "Benchmark test set held out",\n     "check_paths": ["data/03_benchmark/benchmark_test.csv"]},\n    {"phase": "2. Ingestion & Retrieval", "task": "Embedding index built",\n     "check_paths": ["data/05_embeddings_index/stage2_index"]},\n    {"phase": "2. Ingestion & Retrieval", "task": "Retrieval precision/recall evaluated",\n     "check_paths": ["results/stage2/retrieval_metrics.json"]},\n\n    # --- Phase 3: Generation ---\n    {"phase": "3. Generation", "task": "Prompt template + citation format defined",\n     "check_paths": ["code/src/generation/prompt_template.py"]},\n    {"phase": "3. Generation", "task": "Stage 1 (baseline, no retrieval) run complete",\n     "check_paths": ["results/stage1/stage1_baseline_results.json"]},\n    {"phase": "3. Generation", "task": "Stage 2 (+RAG) run complete",\n     "check_paths": ["results/stage2/stage2_rag_results.json"]},\n\n    # --- Phase 4: Verifier ---\n    {"phase": "4. Verifier", "task": "Layer 1 hard citation-existence check implemented",\n     "check_paths": ["code/src/verifier/citation_check.py"]},\n    {"phase": "4. Verifier", "task": "Layer 2 entity-grounding check implemented",\n     "check_paths": ["code/src/verifier/entity_grounding.py"]},\n    {"phase": "4. Verifier", "task": "Injected-error test set built",\n     "check_paths": ["data/03_benchmark/injected_errors.csv"]},\n    {"phase": "4. Verifier", "task": "Stage 3 (+Verifier) run complete",\n     "check_paths": ["results/stage3/stage3_verifier_results.json"]},\n\n    # --- Phase 5: Adaptivity ---\n    {"phase": "5. Adaptivity", "task": "Refresh simulation cases selected",\n     "check_paths": ["data/04_refresh_sim/injected_amendment_cases.csv"]},\n    {"phase": "5. Adaptivity", "task": "Pre/post-refresh index snapshots built",\n     "check_paths": ["data/05_embeddings_index/stage4_post_refresh_index"]},\n    {"phase": "5. Adaptivity", "task": "Stage 4 (+Verifier+Refresh) run complete",\n     "check_paths": ["results/stage4/stage4_refresh_results.json"]},\n\n    # --- Phase 6: Evaluation & Write-up ---\n    {"phase": "6. Evaluation & Write-up", "task": "Evaluation harness built",\n     "check_paths": ["code/src/eval/harness.py"]},\n    {"phase": "6. Evaluation & Write-up", "task": "Human-review calibration done",\n     "check_paths": ["results/human_review_calibration.csv"]},\n    {"phase": "6. Evaluation & Write-up", "task": "Ablation summary table compiled",\n     "check_paths": ["results/ablation_summary_table.csv"]},\n    {"phase": "6. Evaluation & Write-up", "task": "Error analysis notes written",\n     "check_paths": ["results/error_analysis_notes.md"]},\n    {"phase": "6. Evaluation & Write-up", "task": "Plagiarism/originality check run",\n     "check_paths": ["report/plagiarism_report.pdf"]},\n    {"phase": "6. Evaluation & Write-up", "task": "Final report drafted",\n     "check_paths": ["report/final_report.docx"]},\n    {"phase": "6. Evaluation & Write-up", "task": "Presentation deck built",\n     "check_paths": ["report/presentation_deck.pptx"]},\n]\n\n\ndef path_exists_and_nonempty(full_path):\n    if not os.path.exists(full_path):\n        return False\n    if os.path.isdir(full_path):\n        return any(os.scandir(full_path))  # dir exists but must have at least one file\n    return os.path.getsize(full_path) > 0  # file exists and isn\'t a stub\n\n\ndef check_task(root, task):\n    return all(\n        path_exists_and_nonempty(os.path.join(root, p))\n        for p in task["check_paths"]\n    )\n\n\ndef build_report(root):\n    phases = {}\n    for t in TASKS:\n        done = check_task(root, t)\n        phases.setdefault(t["phase"], []).append((t["task"], done, t["check_paths"]))\n    return phases\n\n\ndef print_report(phases):\n    total_tasks = 0\n    total_done = 0\n    lines = []\n    lines.append(f"# Project Progress Report")\n    lines.append(f"_Generated: {datetime.now().isoformat(timespec=\'seconds\')}_\\n")\n\n    for phase, tasks in phases.items():\n        done_count = sum(1 for _, d, _ in tasks if d)\n        pct = 100 * done_count / len(tasks)\n        total_tasks += len(tasks)\n        total_done += done_count\n        lines.append(f"## {phase} — {done_count}/{len(tasks)} ({pct:.0f}%)")\n        for name, done, paths in tasks:\n            mark = "x" if done else " "\n            lines.append(f"- [{mark}] {name}  `({\', \'.join(paths)})`")\n        lines.append("")\n\n    overall_pct = 100 * total_done / total_tasks if total_tasks else 0\n    lines.insert(1, f"**Overall: {total_done}/{total_tasks} tasks complete ({overall_pct:.0f}%)**\\n")\n\n    report_text = "\\n".join(lines)\n    print(report_text)\n    return report_text\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Check IPC2BNS-Verify project progress against the WBS.")\n    parser.add_argument("--root", default=".", help="Path to the project root folder (contains code/, data/, results/, report/)")\n    parser.add_argument("--write-report", action="store_true", help="Also write results/progress_report.md")\n    args = parser.parse_args()\n\n    root = os.path.abspath(args.root)\n    phases = build_report(root)\n    report_text = print_report(phases)\n\n    if args.write_report:\n        out_dir = os.path.join(root, "results")\n        os.makedirs(out_dir, exist_ok=True)\n        out_path = os.path.join(out_dir, "progress_report.md")\n        with open(out_path, "w") as f:\n            f.write(report_text)\n        print(f"\\nSaved report to {out_path}")\n\n\nif __name__ == "__main__":\n    main()\n'

dest_path = os.path.join(PROJECT_ROOT, 'check_progress.py')
with open(dest_path, 'w') as f:
    f.write(check_progress_code)

print(f'✅ check_progress.py written to {dest_path}')

---
## 5. Create Initial Checkpoint (`progress_state.json`)

In [ ]:
import json
from datetime import datetime

checkpoint_path = os.path.join(PROJECT_ROOT, 'checkpoints', 'progress_state.json')

initial_state = {
    'current_phase': 0,
    'completed_phases': [],
    'last_updated': datetime.now().isoformat(timespec='seconds'),
    'next_action': 'Begin Phase 0 setup',
    'notes': 'Project initialized. Directory structure created. Planning docs copied to docs/.'
}

with open(checkpoint_path, 'w') as f:
    json.dump(initial_state, f, indent=2)

print(f'✅ Checkpoint initialized at: {checkpoint_path}')
print()
print(json.dumps(initial_state, indent=2))

---
## 6. Verify Setup — Run `check_progress.py`

In [ ]:
import subprocess

result = subprocess.run(
    ['python', os.path.join(PROJECT_ROOT, 'check_progress.py'),
     '--root', PROJECT_ROOT, '--write-report'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

---
## 7. Visual Directory Tree

In [ ]:
def print_tree(root, prefix='', max_depth=3, current_depth=0):
    """Print directory tree up to max_depth."""
    if current_depth >= max_depth:
        return
    try:
        entries = sorted(os.listdir(root))
    except PermissionError:
        return
    dirs = [e for e in entries if os.path.isdir(os.path.join(root, e))]
    files = [e for e in entries if os.path.isfile(os.path.join(root, e))]
    for f in files:
        print(f'{prefix}📄 {f}')
    for i, d in enumerate(dirs):
        is_last = (i == len(dirs) - 1)
        connector = '└── ' if is_last else '├── '
        print(f'{prefix}{connector}📁 {d}/')
        extension = '    ' if is_last else '│   '
        print_tree(os.path.join(root, d), prefix + extension, max_depth, current_depth + 1)

print(f'📁 NLP_rspaper/')
print_tree(PROJECT_ROOT, '  ')

---
## ✅ Setup Complete!

**What was done:**
1. ✅ Google Drive mounted
2. ✅ Full directory structure created (matches Data Management Plan)
3. ✅ Planning docs copied to `docs/`
4. ✅ `check_progress.py` placed at project root
5. ✅ `checkpoints/progress_state.json` initialized
6. ✅ Progress report generated

---

### What's Next: Phase 0 — Environment & Ground Truth Setup

Phase 0 will produce:
- **Config system** → `code/configs/pipeline_config.yaml`
- **India Code raw text** → `data/00_raw/india_code/` (IPC 1860 + BNS 2023 bare-act text)
- **Concordance source PDFs** → `data/00_raw/concordance_source_pdfs/`
- **Ground-truth concordance table** → `data/02_ground_truth/concordance_v1.csv`

**⏳ Waiting for your confirmation before starting Phase 0.**